In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
En plus de [visualiser les instructions d'un circuit](/guides/visualize-circuits), tu pourrais vouloir visualiser l'ordonnancement d'un circuit en utilisant la méthode Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). Cette visualisation peut t'aider, par exemple, à repérer rapidement les temps d'inactivité sur les qubits. Cependant, cette méthode ne retourne pas de résultats précis pour les circuits dynamiques. Pour visualiser l'ordonnancement d'un circuit dynamique, utilise la méthode `draw_circuit_schedule_timing`, décrite dans la section [Support de Qiskit Runtime](#qr-support).

## Exemples

Pour visualiser un programme de circuit ordonné, tu peux appeler cette fonction avec un ensemble d'arguments de contrôle. La plupart de l'apparence de l'image de sortie peut être modifiée via une feuille de style, mais ce n'est pas obligatoire.

### Afficher avec la feuille de style par défaut

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Afficher avec une feuille de style adaptée au débogage de programmes

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Tu peux créer des fonctions de génération ou de mise en page personnalisées et mettre à jour une feuille de style existante avec ces fonctions. De cette façon, tu peux contrôler la majeure partie de l'apparence de l'image de sortie sans modifier la base de code du dessinateur de circuits ordonnancés. Consulte la référence API [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) pour plus d'exemples.
<span id="qr-support"></span>

## Support de Qiskit Runtime

Bien que le dessinateur de timeline intégré à Qiskit soit utile pour les circuits statiques, il pourrait ne pas refléter avec précision le timing des [circuits dynamiques](/guides/classical-feedforward-and-control-flow), en raison d'opérations implicites telles que la diffusion et la détermination de branches. Dans le cadre de la prise en charge des circuits dynamiques, Qiskit Runtime retourne les informations précises de timing du circuit dans les résultats du job lorsqu'on le demande.

> **Note:** - Il s'agit d'une fonction expérimentale. Elle est en version préliminaire et est donc susceptible d'évoluer.
> - Cette fonction s'applique uniquement aux jobs du Sampler de Qiskit Runtime.
> - Bien que le temps total du circuit soit retourné dans les métadonnées de « compilation », ce n'est PAS le temps utilisé pour la facturation (temps quantique).

### Activer la récupération des données de timing

Pour activer la récupération des données de timing, définis le flag expérimental `scheduler_timing` à `True` lors de l'exécution du job primitif.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Accéder aux données de timing du circuit
Lorsqu'elle est demandée, les données de timing du circuit pour chaque PUB sont retournées dans les métadonnées du résultat du job, sous `["compilation"]["scheduler_timing"]["timing"]`. Ce champ contient les informations de timing brutes. Pour afficher les informations de timing, utilise l'outil de visualisation intégré, décrit dans la section [Visualiser les timings](#visualize-timings).

Utilise le code suivant pour accéder aux données de timing du circuit pour le premier PUB :

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Comprendre les données de timing brutes
Bien que la visualisation des données de timing du circuit à l'aide de la méthode `draw_circuit_schedule_timing` soit le cas d'usage le plus courant, il peut être utile de comprendre la structure des données de timing brutes retournées. Cela peut t'aider, par exemple, à extraire des informations par programmation.

Les données de timing retournées dans `["compilation"]["scheduler_timing"]["timing"]` sont une liste de chaînes de caractères. Chaque chaîne représente une instruction unique sur un canal donné, et est séparée par des virgules selon les types de données suivants :

- `Branch` — Détermine si l'instruction se trouve dans un flux de contrôle (branche then / else) ou dans la branche principale.
- `Instruction` — La porte et le qubit sur lequel opérer.
- `Channel` — Le canal auquel l'instruction est assignée. Il peut s'agir de l'un des canaux suivants :
      - `Qubit x` — Le canal de commande (drive channel) pour le qubit _x_.
      - `AWGRx_y` (générateur de formes d'onde arbitraires en lecture) — Utilisé par les canaux de lecture pour communiquer lors de la mesure des qubits. Les arguments _x_ et _y_ correspondent respectivement à l'identifiant de l'instrument de lecture et au numéro du qubit.
- `T0` — L'heure de début de l'instruction dans l'ordonnancement complet.
- `Duration` — La durée de l'instruction, en unités de secondes _dt_, où 1 dt = 1 cycle d'ordonnancement. Tu peux trouver la valeur `dt` d'un backend en utilisant [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` — Le type d'opération d'impulsion utilisé.

Exemple :

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Visualiser les timings
Avec `qiskit-ibm-runtime` v0.43.0 ou ultérieur, tu peux visualiser les timings des circuits. Pour visualiser les timings, tu dois d'abord convertir les métadonnées du résultat en `fig` à l'aide de la [méthode `draw_circuit_schedule_timing`.](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) Cette méthode retourne une figure `plotly`, que tu peux afficher directement, enregistrer dans un fichier, ou les deux. Pour plus d'informations sur les commandes `plotly` à utiliser, consulte [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) et [`fig.write_image("<path.format>")`.](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Survoler la sortie affiche des informations telles que le début, la fin et la durée.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Exemple de figure générée')
#### Comprendre la figure générée
L'image des données de timing du circuit générée par `draw_circuit_schedule_timing` transmet les informations suivantes :

- L'axe X représente le temps en unités de secondes _dt_, où 1 dt = 1 cycle d'ordonnancement. Tu peux trouver la valeur `dt` d'un backend en utilisant [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- L'axe Y représente le canal (pense aux canaux comme des instruments qui émettent des impulsions).
    - `Receive channel` — Le seul canal qui n'est pas un instrument en soi. Il s'agit d'une instruction exécutée sur tous les canaux faisant partie d'une procédure de communication avec le hub à ce moment-là.
    - `Qubit x` — Le canal de commande (drive channel) pour le qubit x.
    - `AWGRx_y` (générateur de formes d'onde arbitraires en lecture) — Utilisé par les canaux de lecture pour communiquer lors de la mesure des qubits. Les arguments _x_ et _y_ correspondent respectivement à l'identifiant de l'instrument de lecture et au numéro du qubit.
    - `Hub` — Contrôle la diffusion (broadcasting).

De plus, chaque instruction a le format *X_Y*, où *X* est le nom de l'instruction et *Y* est le type d'impulsion. Un type `play` applique des impulsions de contrôle, et un type `capture` enregistre l'état du qubit. Tu peux également survoler chaque instruction pour obtenir plus de détails. Par exemple, la figure précédente montre une impulsion de contrôle pour la porte X appliquée au qubit 10 à 1161 dt.
### Exemple de bout en bout
Cet exemple te montre comment activer l'option, récupérer les informations de timing du circuit depuis les métadonnées, et les afficher sous forme d'image.

D'abord, configure l'environnement, définis les circuits et convertis-les en circuits ISA, puis définis et lance les jobs.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Ensuite, récupère le timing d'ordonnancement du circuit :